# Plain-text NER

Compare small and large GLiNER on the 50 supplied documents. Only aggregate metrics are displayed.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd() if (Path.cwd() / 'ner').is_dir() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
from ner.evaluation import ENTITY_LABELS, evaluate_ner_strict, evaluate_ner_relaxed
from ner.utils import load_jsonl, save_responses_to_jsonl
from ner.inference import DEFAULT_MODEL, LARGE_MODEL, load_gliner_model, predict_dataset
samples = load_jsonl(ROOT / 'data/plain/validation.jsonl')
print('Documents:', len(samples))

In [ ]:
for name, model_name in [('small', DEFAULT_MODEL), ('large', LARGE_MODEL)]:
    output = ROOT / f'results/plain/gliner_{name}.jsonl'
    if not output.exists():
        model = load_gliner_model(model_name)
        predictions = predict_dataset(model, samples, list(ENTITY_LABELS), threshold=0.5)
        save_responses_to_jsonl(predictions, output)
        del model
    print('Model:', model_name)
    strict = evaluate_ner_strict(ROOT / 'data/plain/validation.jsonl', output)
    relaxed = evaluate_ner_relaxed(ROOT / 'data/plain/validation.jsonl', output)

Strict scoring requires the exact start, end and label. Relaxed scoring compares labels and entity texts after whitespace normalization. Occurrences count repeated mentions; unique counts identical mentions once per document. Neither relaxed view ignores case, accents or OCR spelling errors.